# 05 Credit Risk

[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE)

In [ ]:
# === Environment Setup ===
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

# --- Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 14, 'figure.figsize': (12, 8), 'figure.dpi': 150})
%config InlineBackend.figure_format = 'retina'
np.set_printoptions(suppress=True, linewidth=120, precision=4)



### Table of Contents

1.  [Introduction: The Option to Default](#1.-Introduction:-The-Option-to-Default)
2.  [Structural Models: The Merton (1974) Framework](#2.-Structural-Models:-The-Merton-(1974)-Framework)
    - [Equity as a Call Option](#Equity-as-a-Call-Option)
    - [Implementation and Sensitivity Analysis](#Implementation-and-Sensitivity-Analysis)
3.  [Reduced-Form Models: Jarrow-Turnbull (1995)](#3.-Reduced-Form-Models:-Jarrow-Turnbull-(1995))
    - [Hazard Rates and Default Intensity](#Hazard-Rates-and-Default-Intensity)
4.  [Application: Calibrating Merton's Model to Real Data](#4.-Application:-Calibrating-Merton's-Model-to-Real-Data)
    - [The KMV Iterative Algorithm](#The-KMV-Iterative-Algorithm)
    - [Calculating Distance-to-Default (DD)](#Calculating-Distance-to-Default-(DD))
5.  [The Credit Spread Puzzle](#5.-The-Credit-Spread-Puzzle)
6.  [Summary](#6.-Summary)
7.  [Exercises](#7.-Exercises)

# The Lens: Default as a Rational Choice

**What economic problem are we solving?**
When you lend money to a company, you face a risk: they might not pay you back. This is **credit risk**. How do we price this risk? How much extra interest should a risky company pay compared to the US government? To answer this, we need to understand *why* firms default.

**Why do we need this method?**
Default isn't just bad luck; it's an economic decision. Robert Merton's brilliant insight was to treat a firm's equity as a **call option** on its assets. If the assets are worth more than the debt, shareholders pay the debt and keep the difference (exercise the option). If assets are worth less, they walk away (let the option expire). This structural view links stock prices, volatility, and leverage directly to the probability of default, giving us a powerful tool to price corporate bonds and credit default swaps (CDS).

### Learning Objectives
* **Model** equity as a call option on firm assets using the Merton (1974) framework.
* **Compute** the distance-to-default and implied default probabilities from market data.
* **Price** credit default swaps (CDS) and corporate bond spreads.
* **Compare** structural (Merton) and reduced-form (Jarrow-Turnbull) approaches to credit risk.

### Prerequisites
* **Option Pricing:** Black-Scholes formula and put-call parity (Module 09 - Option Pricing).
* **Continuous-Time Finance:** GBM and Ito's Lemma (Module 09 - Continuous-Time Finance).
* **Statistics:** Normal distribution, cumulative distribution functions.

### 1. Introduction: The Option to Default

Credit risk modeling generally falls into two camps:
1.  **Structural Models (Merton):** View default as an endogenous event triggered when firm value falls below a threshold (the debt face value). These models use option pricing theory to value corporate liabilities.
2.  **Reduced-Form Models (Jarrow-Turnbull):** Treat default as an exogenous random event governed by a hazard rate (intensity), similar to how actuaries model mortality.

This notebook focuses on the structural approach, which provides the deepest economic intuition.

### 2. Structural Models: The Merton (1974) Framework

The Merton model assumes a firm has a simple capital structure: equity and a single zero-coupon bond with face value $F$ maturing at time $T$. The firm's asset value $V_A$ follows a Geometric Brownian Motion.

#### Equity as a Call Option
At maturity $T$, the payoff to equity holders is:
$$ E_T = \max(V_A(T) - F, 0) $$
This is exactly the payoff of a **European Call Option** on the firm's assets with strike price $F$. Therefore, we can value the firm's equity using the Black-Scholes formula:
$$ E_0 = V_0 N(d_1) - F e^{-rT} N(d_2) $$

The debt holders get whatever is left. The value of risky debt is:
$$ D_0 = V_0 - E_0 $$
This implies risky debt is equivalent to a risk-free bond *minus* the value of a put option on the firm's assets (the "default put").

In [ ]:
### Merton Model Implementation
def merton_model(V, F, T, r, sigma):
    """Calculates equity, debt, and default probability using Merton's model."""
    d1 = (np.log(V / F) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    # Value of equity (Call Option)
    E = V * norm.cdf(d1) - np.exp(-r * T) * F * norm.cdf(d2)

    # Value of debt (Assets - Equity)
    D = V - E

    # Yield to maturity on the risky debt
    y = -np.log(D/F) / T

    # Credit Spread
    spread = y - r

    # Risk-Neutral Probability of Default (Probability that V_T < F)
    # P(V_T < F) = N(-d2)
    prob_default = norm.cdf(-d2)

    return {'Equity': E, 'Debt': D, 'Yield': y, 'Spread': spread, 'PD': prob_default}

# Example parameters
V0, F_debt, T, r, sigma_A = 100, 80, 1, 0.05, 0.2
results = merton_model(V0, F_debt, T, r, sigma_A)

print("> **Note:** Merton Model Results:")
for k, v in results.items():
    print(f"  {k}: {v:.4f}")

#### Implementation and Sensitivity Analysis
Let's see how credit spreads respond to changes in leverage and asset volatility. This is crucial for understanding why spreads widen during crises (when asset values fall and volatility spikes).

In [ ]:
### Sensitivity Analysis of Credit Spreads

# 1. Sensitivity to Leverage (Debt/Assets)
leverage_ratios = np.linspace(0.1, 0.95, 100)
spreads_lev = [merton_model(100, 100*L, 1, 0.05, 0.2)['Spread'] for L in leverage_ratios]

# 2. Sensitivity to Asset Volatility
volatilities = np.linspace(0.05, 0.6, 100)
spreads_vol = [merton_model(100, 80, 1, 0.05, v)['Spread'] for v in volatilities]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Merton Model Sensitivities", fontsize=18)

ax1.plot(leverage_ratios, spreads_lev, lw=2)
ax1.set_title('a) Credit Spread vs. Leverage')
ax1.set_xlabel('Leverage (Debt/Assets)')
ax1.set_ylabel('Credit Spread')

ax2.plot(volatilities, spreads_vol, lw=2, color='orange')
ax2.set_title('b) Credit Spread vs. Asset Volatility')
ax2.set_xlabel(r'Asset Volatility ($\sigma_A$)')

for ax in [ax1, ax2]: ax.grid(True)
plt.show() # Render plot


### 3. Reduced-Form Models: Jarrow-Turnbull (1995)

Reduced-form models ignore the firm's assets. Instead, they assume default arrives with a certain intensity $\lambda$. The probability of default over a small time $dt$ is $\lambda dt$. The value of a risky bond is:
$$ D(0, T) = F e^{-(r+\lambda(1-R))T} $$
where $R$ is the recovery rate. The credit spread is simply $s = \lambda(1-R)$. These models are easier to calibrate to market data (e.g., the term structure of spreads) but lack the structural economic intuition of Merton.

### 4. Application: Calibrating Merton's Model to Real Data

Merton's model is elegant, but it has a problem: **we cannot observe the market value of a firm's assets ($V_A$) or their volatility ($\sigma_A$).** We only observe the value of Equity ($E$) and its volatility ($\sigma_E$).

To make the model usable, we treat $V_A$ and $\sigma_A$ as unknowns to be solved for. We have two equations:
1.  **Equity Pricing Equation:** $E = \text{Call}(V_A, \sigma_A, F, T, r)$
2.  **Itô's Lemma for Equity Volatility:** The volatility of equity is related to asset volatility by the elasticity of equity value with respect to asset value:
    $$ \sigma_E = \frac{V_A}{E} \Delta_{\text{Call}} \sigma_A = \frac{V_A}{E} N(d_1) \sigma_A $$

This gives us a system of two non-linear equations with two unknowns ($V_A, \sigma_A$). Commercial providers like **KMV (now Moody's Analytics)** use an iterative algorithm to solve this system.

In [ ]:
### The KMV Iterative Algorithm
from scipy.optimize import fsolve


def solve_merton_kmv(E_obs, sigma_E_obs, F, T, r):
    """
    Solves for unobservable Asset Value (V) and Asset Volatility (sigma_A)
    given observable Equity Value (E) and Equity Volatility (sigma_E).
    """
    def system(x):
        V, sigma_A = x
        # Prevent negative values during solver steps
        if V <= 0 or sigma_A <= 0: return [1e6, 1e6]

        d1 = (np.log(V / F) + (r + 0.5 * sigma_A**2) * T) / (sigma_A * np.sqrt(T))
        d2 = d1 - sigma_A * np.sqrt(T)

        # Eq 1: Equity value
        eq1 = (V * norm.cdf(d1) - F * np.exp(-r * T) * norm.cdf(d2)) - E_obs

        # Eq 2: Equity volatility relationship: sigma_E * E = sigma_A * V * N(d1)
        eq2 = (sigma_A * V * norm.cdf(d1)) - (sigma_E_obs * E_obs)

        return [eq1, eq2]

    # Initial guess: V ~ E + F, sigma_A ~ sigma_E * (E/V)
    V_guess = E_obs + F
    sigma_guess = sigma_E_obs * (E_obs / V_guess)

    V_implied, sigma_A_implied = fsolve(system, [V_guess, sigma_guess])
    return V_implied, sigma_A_implied

# Example Calibration
E_market = 50 # Market Cap (e.g., $50B)
sigma_E_market = 0.40 # Annualized Stock Volatility (40%)
Debt_Face = 40 # Face Value of Debt ($40B)
T_debt = 1 # 1 year horizon
r_rate = 0.02

V_calib, sigma_calib = solve_merton_kmv(E_market, sigma_E_market, Debt_Face, T_debt, r_rate)

print("> **Note:** Calibration Results:")
print(f"  Implied Asset Value (V_A): ${V_calib:.2f}B")
print(f"  Implied Asset Volatility (sigma_A): {sigma_calib:.2%}")

# Calculate Distance to Default
DD = (V_calib - Debt_Face) / (V_calib * sigma_calib)
print(f"  Distance-to-Default (DD): {DD:.2f} standard deviations")

#### Calculating Distance-to-Default (DD)
The **Distance-to-Default (DD)** is the most popular metric from the KMV model. It measures how many standard deviations the asset value is away from the default barrier (debt face value). 
$$ \text{DD} = \frac{V_A - F}{V_A \sigma_A} $$
A higher DD implies a safer firm. This metric is widely used by banks and hedge funds to rank companies by creditworthiness.

### 5. The Credit Spread Puzzle

While the Merton model is brilliant, it consistently underpredicts actual market credit spreads. This is the **credit spread puzzle**. Even for very safe firms with almost zero default probability in the model, market spreads are significantly positive.

**Why?**
1.  **Liquidity:** Corporate bonds are less liquid than Treasuries. Investors demand a premium for this illiquidity, which the Merton model interprets as default risk.
2.  **Jumps:** Asset prices can jump downwards (e.g., due to fraud or sudden regulation). GBM assumes continuous paths, underestimating the risk of sudden default.
3.  **Taxes:** Corporate interest payments are tax-deductible, which affects bond pricing.

# Summary

**What did we learn?**
- **Default as an Option:** Merton's structural model treats equity as a call option on the firm's assets. Default occurs rationally when assets < debt.
- **Linkage:** This framework links equity markets (stock price, volatility) to credit markets (bond yields, spreads). A rise in stock volatility increases the value of equity (option) but decreases the value of debt.
- **Calibration:** Since asset values are unobservable, we use the KMV algorithm to back them out from observable equity data.
- **Distance-to-Default:** A key output of the calibrated model, providing a normalized measure of credit health.

**What is the key takeaway?**
Credit risk is not a separate silo; it is derivatives pricing in disguise. By viewing the capital structure through the lens of options, we gain a unified understanding of corporate value.

### 7. Exercises

1.  **Equity as Volatility:** In the Merton model, if a firm's asset volatility increases (all else equal), what happens to the value of its equity? What happens to the value of its debt? Explain the intuition using option theory (Vega).

2.  **KMV Algorithm:** Why do we need an iterative solver for the KMV model? Why can't we just observe the volatility of the firm's assets directly from its balance sheet?

3.  **Subordinated Debt:** Suppose a firm has two layers of debt: Senior ($F_1$) and Junior ($F_2$). How would you model the Junior debt as an option? (Hint: It pays off only if $V > F_1$. It's like a call spread).

4.  **Default Probability:** Using the calibrated values from the case study, calculate the real-world probability of default. How does this compare to the risk-neutral probability $N(-d_2)$? Which one should be higher?

In [ ]:
# --- Merton Structural Model for Credit Risk ---

def merton_model(V, D, T, r, sigma_V):
    """
    Calculate Equity Value and Probability of Default using Merton (1974).
    V: Value of Firm's Assets
    D: Face Value of Debt
    T: Time to Maturity
    r: Risk-free rate
    sigma_V: Volatility of Asset Value
    """
    # d1 and d2 from Black-Scholes
    d1 = (np.log(V / D) + (r + 0.5 * sigma_V**2) * T) / (sigma_V * np.sqrt(T))
    d2 = d1 - sigma_V * np.sqrt(T)

    # Equity Value (Call Option on Assets)
    E = V * norm.cdf(d1) - D * np.exp(-r * T) * norm.cdf(d2)

    # Probability of Default (Risk-Neutral)
    # Default occurs if V_T < D
    # P(V_T < D) = N(-d2)
    PD = norm.cdf(-d2)

    # Credit Spread (in basis points)
    # Yield = -ln(Price/Face)/T
    # Price of Debt = D * exp(-rT) - Put (Put is value of default option)
    # Or simpler: Value of Debt = V - E
    B = V - E
    yield_debt = -np.log(B/D) / T
    spread = (yield_debt - r) * 10000

    return E, PD, spread

# Parameters
V_0 = 100    # Asset Value
D = 80       # Debt Face Value
T = 1.0      # Maturity (1 year)
r = 0.05     # Risk-free rate
sigma_V = 0.2 # Asset Volatility

E, PD, spread = merton_model(V_0, D, T, r, sigma_V)

print(f"Firm Asset Value: {V_0}")
print(f"Debt Face Value:  {D}")
print("-" * 30)
print(f"Equity Value:             {E:.4f}")
print(f"Probability of Default:   {PD:.4%}")
print(f"Credit Spread:            {spread:.2f} bps")

# Visualizing PD vs Leverage
leverage_ratios = np.linspace(0.5, 1.5, 50) # D/V
pds = []
for lev in leverage_ratios:
    # Fix V, vary D
    D_i = V_0 * lev
    _, pd_i, _ = merton_model(V_0, D_i, T, r, sigma_V)
    pds.append(pd_i)

plt.figure(figsize=(8, 5))
plt.plot(leverage_ratios, pds) # Plot data series
plt.xlabel("Leverage Ratio (Debt/Assets)")
plt.ylabel("Probability of Default")
plt.title("Merton Model: PD vs Leverage")
plt.grid(True, alpha=0.3)
plt.show() # Render plot


---
## Summary

In this lecture, we have systematically explored the theoretical and practical aspects of the model.

**Key Takeaways:**
1.  **Foundations:** We established the mathematical basis of the economic problem.
2.  **Computation:** We implemented the solution using efficient algorithms.
3.  **Implications:** We analyzed the economic significance of the results.

**Further Exploration:**
- Experiment with model parameters to assess sensitivity.
- Extend the framework by relaxing simplifying assumptions.